# Sumas, promedios y el teorema central del límite

El capítulo anterior trabajó con **una variable aleatoria a la vez**. Pero
los datos reales casi siempre son sumas o promedios: el total de defectos
del día, la proporción observada de «sí» en la encuesta, el tiempo total que
pasó la clínica esperando, el promedio de medias musculares en un estudio.

Este capítulo responde dos preguntas centrales:

1. **LLN — ley de los grandes números.** ¿A qué tiende el promedio cuando
   $n$ crece?
2. **TCL — teorema central del límite.** ¿Cómo se distribuye ese promedio
   alrededor del límite?


## Preguntas de inicio

1. La clínica registra 30 esperas en un día. ¿Cómo se distribuye el
   **promedio** de esas 30 esperas? ¿Y el **total**?
2. La encuesta del Hilo B sale a la calle con una muestra grande. ¿Por qué la
   proporción observada es cada vez más confiable?
3. ¿Por qué tantos fenómenos parecen Normales aunque la fuente no lo sea?
4. Si la cantidad de defectos por turno es Bin(100, 0,4), ¿cómo aproximamos
   $P(Y \le 45)$ sin tabular las 100 probabilidades de la Binomial?


## Importaciones

In [ ]:
import math

import altair as alt
import pandas as pd

from core import (
    BinomialParams,
    ExponentialParams,
    NormalParams,
    Settings,
)
from distributions import (
    make_binomial,
    make_exponential,
    make_normal,
    tail_probability_of_continuous,
)
from distributions.evaluations import TailProbabilityInput
from exercises import NumericAnswerInput, verify_numeric_answer
from sampling import (
    CLTSimulationInput,
    GaltonBoardInput,
    LLNSimulationInput,
    simulate_clt,
    simulate_galton_board,
    simulate_lln,
)
from visualization import (
    CLTComparisonChartInput,
    LLNChartInput,
    chart_clt_comparison,
    chart_lln_running_mean,
)
from visualization.theme import apply_theme
from widgets import (
    CLTExplorerInput,
    LLNExplorerInput,
    build_clt_explorer,
    build_lln_explorer,
)

In [ ]:
settings = Settings()

## Hilo B: la proporción observada se estabiliza

Volvemos al esquema Bernoulli del Capítulo 3 (ecuación 3.1 con $n = 1$).
Cada respuesta de la encuesta es $X_i \in \{0, 1\}$ con probabilidad
$p$ de ser «sí». El promedio acumulado tras $n$ respuestas es:

$$ \bar{X}_n = \frac{1}{n}\sum_{i=1}^{n} X_i \tag{4.1} $$

Esta es la misma definición que (1.1) del Capítulo 1, pero ahora con $X_i$
**variables aleatorias**, no observaciones fijas.

**LLN (ley de los grandes números, versión débil):** si las $X_i$ son
i.i.d. con $E[X_i] = \mu$ y varianza finita,

$$ \bar{X}_n \xrightarrow{P} \mu \quad\text{cuando } n \to \infty \tag{4.2} $$


In [ ]:
bernoulli = make_binomial(BinomialParams(trials=1, success_probability=0.55))
lln_input = LLNSimulationInput(
    distribution=bernoulli,
    horizon=4_000,
    settings=settings,
)
lln_result = simulate_lln(lln_input)

lln_chart_input = LLNChartInput(
    lln_result=lln_result,
    title="Encuesta — proporción observada de «sí» (LLN)",
    settings=settings,
)
chart_lln_running_mean(lln_chart_input)

Las primeras 50 respuestas saltan mucho; al promediar 4.000, una respuesta
extra mueve la media en una cantidad despreciable. Esa es la intuición que
formaliza la fórmula (4.2).


In [ ]:
lln_explorer_input = LLNExplorerInput(settings=settings)
build_lln_explorer(lln_explorer_input)

## Independiente: el tablero de Galton

El **tablero de Galton** es la metáfora pictórica del TCL. Cada bola toma
$r$ decisiones independientes izquierda/derecha. La posición final es una
**suma** de $r$ pasos de $\pm 1$, y se distribuye aproximadamente Normal —
aunque cada paso individual sea uniforme. Es un ejemplo totalmente distinto
a los hilos del libro y por eso es tan útil para entender el TCL aislado de
tiempos, encuestas o defectos.


In [ ]:
galton_input = GaltonBoardInput(rows=24, balls=8_000, settings=settings)
galton_result = simulate_galton_board(galton_input)

galton_table = pd.DataFrame({
    "posicion": galton_result.bin_positions,
    "frecuencia": galton_result.bin_counts,
})
galton_chart = (
    alt
    .Chart(galton_table)
    .mark_bar(opacity=0.85)
    .encode(
        x=alt.X("posicion:O", title="Casilla final"),
        y=alt.Y("frecuencia:Q", title="Cantidad de bolas"),
    )
    .properties(title="Tablero de Galton — suma de 24 pasos ±1")
)
apply_theme(galton_chart, settings)

## El TCL formal

Sean $X_1, X_2, \dots$ i.i.d. con $E[X_i] = \mu$ y
$\mathrm{Var}(X_i) = \sigma^2 < \infty$. **Paso 1:** centramos el promedio
(4.1) restándole su esperanza $\mu$. **Paso 2:** lo escalamos por el desvío
estándar de $\bar{X}_n$, que es $\sigma/\sqrt{n}$. **Paso 3:** afirmamos:

$$ \begin{aligned}
Z_n &= \frac{\bar{X}_n - \mu}{\sigma/\sqrt{n}} \\[4pt]
Z_n &\xrightarrow{d} \mathcal{N}(0, 1)
\end{aligned} \tag{4.3} $$

Notar que (4.3) **no** dice que $X_i$ tienda a una Normal. Dice que la
**media estandarizada** lo hace.


## Hilo A: TCL aplicado a tiempos de espera

Cada espera es Exponencial (Capítulo 3, ecuación 3.6), claramente **no**
Normal: tiene cola larga y es asimétrica. Pero el promedio de 30 esperas se
vuelve aproximadamente Normal por (4.3). Lo materializamos simulando muchas
réplicas:


In [ ]:
clinic_distribution = make_exponential(ExponentialParams(rate=0.25))
clt_input = CLTSimulationInput(
    distribution=clinic_distribution,
    sample_size_per_replicate=30,
    replicates=5_000,
    settings=settings,
)
clinic_clt_result = simulate_clt(clt_input)

clt_chart_input = CLTComparisonChartInput(
    clt_result=clinic_clt_result,
    title="Promedio de 30 esperas (clínica) — convergencia a la Normal",
    settings=settings,
)
chart_clt_comparison(clt_chart_input)

Las medias estandarizadas de 5.000 días simulados se alinean con la
$\mathcal{N}(0, 1)$ del Capítulo 3 (densidad 3.2). El TCL (4.3) **predice**
esta forma, no la impone artificialmente.


In [ ]:
clt_explorer_input = CLTExplorerInput(settings=settings)
build_clt_explorer(clt_explorer_input)

## Hilo C: aproximación Binomial → Normal

Si $Y \sim \text{Bin}(n, p)$, podemos escribir $Y$ como suma de $n$
Bernoullis i.i.d. Aplicando (4.3) (con $\mu = p$ y $\sigma^2 = p(1-p)$)
y multiplicando por $n$:

$$ \begin{aligned}
Y       &\approx \mathcal{N}\!\bigl(np,\ np(1-p)\bigr) \\[4pt]
        &\quad\text{válida si } np \ge 10 \text{ y } n(1-p) \ge 10
\end{aligned} \tag{4.4} $$

Para los defectos del Hilo C con $n = 100$ y $p = 0{,}4$:


In [ ]:
factory_normal_approx = make_normal(NormalParams(mean=40.0, standard_deviation=math.sqrt(24.0)))
factory_tail_input = TailProbabilityInput(
    distribution=factory_normal_approx,
    upper_bound=45.0,
)
factory_tail_probability = tail_probability_of_continuous(factory_tail_input)
factory_tail_probability

## Ejercicio 1 — Desvío del promedio

Si $X_i \sim \mathcal{N}(50, 100)$ (es decir $\sigma = 10$) y tomamos
$n = 25$ observaciones, ¿cuál es el desvío estándar de $\bar{X}_n$?

**Idea.** Aplicamos el factor de escala que aparece en (4.3):

$$ \sigma_{\bar{X}_n} = \frac{\sigma}{\sqrt{n}} = \frac{10}{\sqrt{25}} = 2 $$


In [ ]:
expected_standard_error = 10.0 / math.sqrt(25)

student_answer_se = 2.0
verify_input = NumericAnswerInput(
    student_answer=student_answer_se,
    expected_answer=expected_standard_error,
)
verify_numeric_answer(verify_input)

## Ejercicio 2 — Aproximar Binomial con Normal

$Y \sim \text{Bin}(100, 0{,}4)$. Aproximá $P(Y \le 45)$ aplicando (4.4)
(sin corrección por continuidad para simplificar):

$$ \begin{aligned}
E[Y]              &= np = 40 \\[4pt]
\mathrm{Var}(Y)   &= np(1-p) = 24 \\[4pt]
P(Y \le 45)       &\approx P\!\left(Z \le \frac{45 - 40}{\sqrt{24}}\right) \\[4pt]
                  &\approx P(Z \le 1{,}02) \approx 0{,}846
\end{aligned} $$


In [ ]:
exercise_normal_approx = make_normal(NormalParams(mean=40.0, standard_deviation=math.sqrt(24.0)))
exercise_tail_input = TailProbabilityInput(
    distribution=exercise_normal_approx,
    upper_bound=45.0,
)
expected_probability = tail_probability_of_continuous(exercise_tail_input).probability

student_answer_probability = 0.846
verify_input = NumericAnswerInput(
    student_answer=student_answer_probability,
    expected_answer=expected_probability,
    absolute_tolerance=5e-3,
)
verify_numeric_answer(verify_input)

## Síntesis y respuestas

1. **Distribución del promedio de 30 esperas.** Por (4.3), está
   aproximadamente centrada en $\mu = 4$ minutos con desvío
   $\sigma/\sqrt{30} \approx 0{,}73$ minutos. La forma es Normal aunque
   cada espera sea Exponencial.
2. **¿Por qué la encuesta es más confiable?** Por (4.2) y (4.3): la
   proporción observada se acerca a $p$ y su desvío baja como
   $\sqrt{p(1-p)/n}$.
3. **¿Por qué tantos fenómenos parecen Normales?** Porque muchos son sumas o
   promedios de efectos chicos e independientes; (4.3) **predice** la forma
   Normal del agregado.
4. **Aproximar Bin(100, 0,4) con Normal.** Por (4.4),
   $Y \approx \mathcal{N}(40, 24)$. Eso reduce $P(Y \le 45)$ a una cuenta
   sobre la Normal estándar — `factory_tail_probability` en este capítulo.


## Próximas preguntas

Tenemos modelos para los datos y herramientas para sus promedios. Lo que
falta es **invertir** el problema: dado lo que vimos, ¿qué podemos decir de
los parámetros desconocidos?

- ¿Cuál es el rango plausible para la espera media de la clínica?
- ¿Qué intervalo cubre la verdadera proporción de la encuesta?
- ¿Podemos rechazar la afirmación de la fábrica de que los defectos están
  bajo el 5%?

Ese es el oficio del próximo capítulo: **inferencia**.
